# Get company name from text



In [16]:
import re

DATE_PREFIX_RE = re.compile(
    r"""^\s*
        (?:                                   # common date prefixes
            \d{1,2}\s*[A-Za-z]{3,}\w*\s*\d{2,4}   # 13 febv 26 / 17 Feb 2026
          | \d{1,2}[-/]\d{1,2}[-/]\d{2,4}         # 13-02-2026 / 13/02/26
        )
        \s*[-–—:]*\s*                          # optional separators after date
    """,
    re.IGNORECASE | re.VERBOSE
)

def extract_company_name_from_bse_text(text: str) -> str | None:
    # Find first 6-digit BSE scrip code
    m = re.search(r"\b\d{6}\b", text)
    if not m:
        return None
    company_code = m.group(0)

    # Take only the same-line chunk immediately before the code
    line_start = max(text.rfind("\n", 0, m.start()), text.rfind("\r", 0, m.start()))
    line_start = 0 if line_start == -1 else line_start + 1
    candidate = text[line_start:m.start()].strip()

    # Remove a leading date prefix if present (e.g., "13 febv 26- ")
    candidate = DATE_PREFIX_RE.sub("", candidate).strip()

    # Clean trailing separators / currency artifacts
    candidate = re.sub(r"[-\s]+$", "", candidate)
    candidate = re.sub(r"[-\s]*[₹$]+$", "", candidate)
    candidate = re.sub(r"[-\s]+$", "", candidate).strip()

    if candidate:
        return company_code + "_" + candidate
    elif company_code:
        return company_code + "_"
    else:
        return text.strip()  # fallback to raw text if no code found


tests = [
"13 febv 26 \n Desco Infratech Ltd - 544387 - Announcement under Regulation 30 (LODR)-Award_of_Order_Receipt_of_Order",
"13 febv 26- Gabion Technologies India Ltd - 544675 - Announcement under Regulation 30 (LODR)-Award_of_Order_Receipt_of_Order",
"HFCL Ltd - 500183 - Announcement under Regulation 30 (LODR)-Award_of_Order_Receipt_of_Order",
"Highway Infrastructure Ltd - 544477 - Announcement under Regulation 30 (LODR)-Award_of_Order_Receipt_of_Order",
"Seamec Ltd - 526807 - Announcement under Regulation 30 (LODR)-Award_of_Order_Receipt_of_Order",
"17 Feb 2026\nHazoor Multi Projects Ltd-$ - 532467 - Announcement under Regulation 30 (LODR)-Award_of_Order_Receipt_of_Order ..."
]

for t in tests:
    print(extract_company_name_from_bse_text(t))

544387_Desco Infratech Ltd
544675_Gabion Technologies India Ltd
500183_HFCL Ltd
544477_Highway Infrastructure Ltd
526807_Seamec Ltd
532467_Hazoor Multi Projects Ltd


# get company data

In [59]:
import os
from datetime import datetime, timedelta

def get_pdf_files(before_days: int = 0):
    # Get today's date in yyyy-mm-dd format
    today_date = datetime.now().strftime('%Y-%m-%d')


    # Subtract n day using timedelta
    days_back = datetime.now() - timedelta(before_days)

    # Format the resulting datetime object into a YYYY-MM-DD string
    days_back_formatted = days_back.strftime('%Y-%m-%d')

    # Path to the folder
    folder_path = f'bse_award_orders/{days_back_formatted}'
    
    # List all files in the folder
    files = os.listdir(folder_path)
    
    # Filter to get only the PDF files
    pdf_files = [file for file in files if file.endswith('.pdf')]
    
    return pdf_files

pdf_files = get_pdf_files(before_days=0)
print(pdf_files)

['543806_ITCONS E-Solutions Ltd.pdf']


In [ ]:
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

class ScreenerAutomation:
    def __init__(self):
        # Initialize WebDriver (This automatically uses the ChromeDriver installed)
        self.driver = webdriver.Chrome()

    def fetch_company_data(self, company_code, company_name):
        # Try to open company page
        url = f"https://www.screener.in/company/{company_code}/"
        self.driver.get(url)

        try:
            # Wait for page to load or give a timeout
            WebDriverWait(self.driver, 10).until(EC.presence_of_element_located((By.TAG_NAME, "h1")))

            # Check for 404 error by looking for page elements
            if "404 Not Found" in self.driver.title:
                # Page not found, search for the company
                self.driver.get("https://www.screener.in/")
                search_box = self.driver.find_element(By.ID, "id_search_input")
                search_box.send_keys(company_name)
                search_box.send_keys(Keys.RETURN)

                # Wait for the search results to load and click the first result
                WebDriverWait(self.driver, 10).until(EC.presence_of_element_located((By.CSS_SELECTOR, "ul#search-results li a")))
                first_result = self.driver.find_element(By.CSS_SELECTOR, "ul#search-results li a")
                first_result.click()

                # Wait for the company page to load
                WebDriverWait(self.driver, 10).until(EC.presence_of_element_located((By.TAG_NAME, "h1")))

        except Exception as e:
            print(f"Error occurred while fetching data for {company_name} with code {company_code}: {e}")
            return None

        # Now extract data from the page
        return self.extract_company_data()

    def extract_company_data(self):
        try:
            # Ensure we are on the "Profit & Loss" tab
            profit_loss_tab = self.driver.find_element(By.XPATH, "//a[contains(text(), 'Profit & Loss')]")
            profit_loss_tab.click()  # Click to focus on the "Profit & Loss" tab

            # Wait for the "Profit & Loss" section to load
            WebDriverWait(self.driver, 10).until(EC.presence_of_element_located((By.XPATH, "//table[contains(@class, 'data-table')]")))

            # Locate the "Net Profit" expand button
            expand_button = self.driver.find_element(By.XPATH, "//button[contains(text(), 'Net Profit')]")
            
            # Scroll into view to make sure it is clickable
            self.driver.execute_script("arguments[0].scrollIntoView(true);", expand_button)

            # Wait to ensure it is clickable
            WebDriverWait(self.driver, 10).until(EC.element_to_be_clickable(expand_button))
            
            # Use JavaScript to click if normal click is blocked
            self.driver.execute_script("arguments[0].click();", expand_button)

            # Wait for the page to update and the table to load with the necessary data
            time.sleep(3)  # Sleep for a few seconds to let the data load

            # Get the HTML of the entire page
            html = self.driver.page_source

            # Pass the HTML to BeautifulSoup for parsing
            soup = BeautifulSoup(html, "html.parser")
            # Refined XPath queries
            market_cap = self.driver.find_element(By.XPATH, '//*[contains(text(), "Market Cap")]/following-sibling::span').text
            roce = self.driver.find_element(By.XPATH, '//*[contains(text(), "ROCE")]/following-sibling::span').text
            roe = self.driver.find_element(By.XPATH, '//*[contains(text(), "ROE")]/following-sibling::span').text
            # net_profit_ttm = self.driver.find_element(By.XPATH, '/html/body/main/section[4]/div[3]/table/tbody/tr[1]/td[14]').text
            # Now extract the Net Profit TTM value after expanding the section
            # net_profit_ttm = self.driver.find_element(By.XPATH, '//tr[contains(@class, "strong")]//td[contains(text(), "Net Profit")]/following-sibling::td[last()]').text

            # Return extracted data as a dictionary
            return {
                "Market Cap": market_cap,
                "ROCE": roce,
                "ROE": roe,
                # "Net Profit TTM": net_profit_ttm
            }
        except Exception as e:
            print(f"Error while extracting company data: {e}")
            return None

    def close(self):
        # Close the browser once done
        self.driver.quit()
import os
from datetime import datetime

# Step 1: Get all PDF files from the folder

# Step 2: Process each file and extract company data
def process_pdf_files(pdf_files):
    screener = ScreenerAutomation()
    
    for pdf_file in pdf_files:
        # Extract company code and company name from PDF filename
        company_code, company_name = pdf_file.replace('.pdf', '').split('_')
        
        print(f"Processing {company_name} with code {company_code}")
        company_data = screener.fetch_company_data(company_code, company_name)
        
        if company_data:
            print(f"Company Data for {company_name}: {company_data}")
        else:
            print(f"Failed to retrieve data for {company_name}")
    
    # Close the browser once all data is processed
    screener.close()

# Get PDF files and process them
pdf_files = get_pdf_files(before_days=1)
process_pdf_files(pdf_files[:2])